In [3]:
#showing a simple transformer architecture

import torch
import torch.nn as nn
import torch.nn.functional as F

#self attention module

class SelfAttention(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        #linear vector creation
        self.query = nn.Linear(embed_size, embed_size)
        self.key = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)
        
    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        scale = x.shape[-1] ** 0.5
        scores = Q @ K.transpose(-2, -1) / scale
        attention_weights = F.softmax(scores, dim=-1)
        return attention_weights @ V

In [4]:
#Transfomer encoder
class TransformerBlock(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        self.attention = SelfAttention(embed_size)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, embed_size * 4),
            nn.ReLU(),
            nn.Linear(embed_size * 4, embed_size)
        )
        
    def forward(self, x):
        attention_out = self.attention(x)
        x = self.norm1(attention_out + x)
        ff_out = self.feed_forward(x)
        return self.norm2(ff_out + x)

In [5]:
embed_size = 256
sequence_length = 10
batch_size = 32

model = TransformerBlock(embed_size)

dummy_input = torch.randn(batch_size, sequence_length, embed_size)

output = model(dummy_input)

print(output.shape)

torch.Size([32, 10, 256])
